<a href="https://colab.research.google.com/github/Kush-Singh-26/NLP/blob/main/Seq2Seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Seq2Seq

## Importing the libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import spacy
import tqdm


## For uniformity

In [2]:
seed = 1234

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

# Loading the data

- Data is downloaded from [here](https://huggingface.co/datasets/bentrevett/multi30k)
- Using Hugging Face datasets

In [3]:
import jsonlines
from datasets import Dataset, DatasetDict

def load_jsonl(file_path):
    data = []
    with jsonlines.open(file_path) as reader:
        for obj in reader:
            data.append(obj)
    return Dataset.from_list(data)

# Load each split
train_dataset = load_jsonl("data/train.jsonl")
val_dataset = load_jsonl("data/val.jsonl")
test_dataset = load_jsonl("data/test.jsonl")

# Combine into a DatasetDict
full_dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})


In [4]:
train_dataset[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'}

## Using `spacy` for tokenization

In [5]:
en_nlp = spacy.load("en_core_web_sm")
de_nlp = spacy.load("de_core_news_sm")

In [6]:
string = "What a lovely day it is today!"

[token.text for token in en_nlp.tokenizer(string)]

['What', 'a', 'lovely', 'day', 'it', 'is', 'today', '!']

In [7]:
def tokenize_example(example, en_nlp, de_nlp, max_length, lower, sos_token, eos_token):
    en_tokens = [token.text for token in en_nlp.tokenizer(example["en"])][:max_length]
    de_tokens = [token.text for token in de_nlp.tokenizer(example["de"])][:max_length]
    if lower:
        en_tokens = [token.lower() for token in en_tokens]
        de_tokens = [token.lower() for token in de_tokens]
    en_tokens = [sos_token] + en_tokens + [eos_token]
    de_tokens = [sos_token] + de_tokens + [eos_token]
    return {"en_tokens": en_tokens, "de_tokens": de_tokens}

In [8]:
max_length = 1_000
lower = True
sos_token = "<sos>"
eos_token = "<eos>"

fn_kwargs = {
    "en_nlp": en_nlp,
    "de_nlp": de_nlp,
    "max_length": max_length,
    "lower": lower,
    "sos_token": sos_token,
    "eos_token": eos_token,
}

# Apply map directly to the datasets.Dataset objects from the DatasetDict
train_data = full_dataset["train"].map(tokenize_example, fn_kwargs=fn_kwargs)
valid_data = full_dataset["validation"].map(tokenize_example, fn_kwargs=fn_kwargs)
test_data = full_dataset["test"].map(tokenize_example, fn_kwargs=fn_kwargs)

Map:   0%|          | 0/29000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1014 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [9]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

## Building Vocabulary

In [10]:
from collections import Counter

class Vocabulary:
    def __init__(self, min_freq=1, specials=None):
        """
        Parameters:
        - min_freq: Minimum frequency a word must have to be included.
        - specials: List of special tokens like ["<unk>", "<pad>", "<sos>", "<eos>"]
        """
        self.min_freq = min_freq
        self.word_counts = Counter()
        self.specials = specials if specials else []

        self.itos = {}
        self.stoi = {}

        # Reserve space for special tokens at the beginning
        for idx, token in enumerate(self.specials):
            self.itos[idx] = token
            self.stoi[token] = idx

    def __getitem__(self, token):
        return self.stoi.get(token, self.stoi['<unk>'])

    def __len__(self):
        return len(self.itos)

    def add_sentence(self, tokens):
        self.word_counts.update(tokens)

    def build_vocabulary(self, iterator):
        """
        iterator: a list of tokenized sentences (list of list of strings)
        """
        for tokens in iterator:
            self.add_sentence(tokens)

        idx = len(self.itos)  # Start indexing after special tokens

        for word, freq in self.word_counts.items():
            if freq >= self.min_freq and word not in self.stoi:
                self.stoi[word] = idx
                self.itos[idx] = word
                idx += 1

    def numericalize(self, tokens):
        unk_idx = self.stoi.get("<unk>", 0)  # fallback if <unk> not defined
        return [self.stoi.get(token, unk_idx) for token in tokens]

    def get_itos(self):
        return [self.itos[i] for i in range(len(self.itos))]

    def get_stoi(self):
        return [self.stoi[i] for i in range(len(self.stoi))]

    def lookup_tokens(self, indices):
        return [self.itos.get(index, '<unk>') for index in indices]

    def lookup_indices(self, tokens):
        return [self.stoi.get(token, self.stoi['<unk>']) for token in tokens]


In [11]:
min_freq = 2
unk_token = "<unk>"
pad_token = "<pad>"
sos_token = "<sos>"
eos_token = "<eos>"

special_tokens = [unk_token, pad_token, sos_token, eos_token]

# Create English vocab
en_vocab = Vocabulary(min_freq=min_freq, specials=special_tokens)
en_vocab.build_vocabulary(train_data["en_tokens"])  # list of tokenized English sentences

# Create German vocab
de_vocab = Vocabulary(min_freq=min_freq, specials=special_tokens)
de_vocab.build_vocabulary(train_data["de_tokens"])  # list of tokenized German sentences

In [12]:
de_vocab.get_itos()[:10]

['<unk>',
 '<pad>',
 '<sos>',
 '<eos>',
 'zwei',
 'junge',
 'weiße',
 'männer',
 'sind',
 'im']

- Ensure the special tokens of both vocabulary map to same indices

In [13]:
assert en_vocab[unk_token] == de_vocab[unk_token]
assert en_vocab[pad_token] == de_vocab[pad_token]

unk_index = en_vocab[unk_token]
pad_index = en_vocab[pad_token]


In [14]:
tokens = ["i", "love", "watching", "crime", "shows"]

In [15]:
en_vocab.lookup_indices(tokens)

[171, 4010, 225, 0, 1130]

In [16]:
def numericalize_example(example, en_vocab, de_vocab):
    en_ids = en_vocab.lookup_indices(example["en_tokens"])
    de_ids = de_vocab.lookup_indices(example["de_tokens"])
    return {"en_ids": en_ids, "de_ids": de_ids}

In [17]:
fn_kwargs = {"en_vocab": en_vocab, "de_vocab": de_vocab}

train_data_numericalized = train_data.map(numericalize_example, fn_kwargs=fn_kwargs)
valid_data_numericalized = valid_data.map(numericalize_example, fn_kwargs=fn_kwargs)
test_data_numericalized = test_data.map(numericalize_example, fn_kwargs=fn_kwargs)

print(train_data_numericalized[0])
print(type(train_data_numericalized[0]["en_ids"]))


Map:   0%|          | 0/29000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1014 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

{'en': 'Two young, White males are outside near many bushes.', 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.', 'en_tokens': ['<sos>', 'two', 'young', ',', 'white', 'males', 'are', 'outside', 'near', 'many', 'bushes', '.', '<eos>'], 'de_tokens': ['<sos>', 'zwei', 'junge', 'weiße', 'männer', 'sind', 'im', 'freien', 'in', 'der', 'nähe', 'vieler', 'büsche', '.', '<eos>'], 'en_ids': [2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 3], 'de_ids': [2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 3]}
<class 'list'>


- The `with_format` method converts features indicated by the `columns` argument to a given `type`.

In [18]:
data_type = "torch"
format_columns = ["en_ids", "de_ids"]

# Apply with_format to the numericalized datasets
train_data = train_data_numericalized.with_format(
    type=data_type, columns=format_columns, output_all_columns=True
)

valid_data = valid_data_numericalized.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

test_data = test_data_numericalized.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

In [19]:
def get_collate_fn(pad_index):
    def collate_fn(batch):
        batch_en_ids = [example["en_ids"] for example in batch]
        batch_de_ids = [example["de_ids"] for example in batch]
        batch_en_ids = nn.utils.rnn.pad_sequence(batch_en_ids, padding_value=pad_index)
        batch_de_ids = nn.utils.rnn.pad_sequence(batch_de_ids, padding_value=pad_index)
        batch = {
            "en_ids": batch_en_ids,
            "de_ids": batch_de_ids,
        }
        return batch

    return collate_fn

In [20]:
def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
    collate_fn = get_collate_fn(pad_index)
    data_loader = torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=shuffle,
    )
    return data_loader

In [21]:
batch_size = 128

train_data_loader = get_data_loader(train_data, batch_size, pad_index, shuffle=True)
valid_data_loader = get_data_loader(valid_data, batch_size, pad_index)
test_data_loader = get_data_loader(test_data, batch_size, pad_index)

## Encoder

In [22]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(input_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded) # embedded = [src length, batch size, embedding dim]

        return hidden, cell


In [23]:
class Decoder(nn.Module):
    def __init__(self, output_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(output_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        # input should be 1D: [batch_size] or 2D: [1, batch_size]
        if input.dim() == 2:
            input = input.squeeze(0)  # make input shape [batch_size]

        input = input.unsqueeze(0)  # now [1, batch_size]

        embedded = self.dropout(self.embedding(input))  # [1, batch_size, embed_dim]

        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))  # LSTM expects 3D
        prediction = self.fc_out(output.squeeze(0))  # [batch_size, output_dim]
        return prediction, hidden, cell


In [24]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio):
        # src = [src length, batch size]
        # trg = [trg length, batch size]
        batch_size = trg.shape[1]
        trg_length = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(trg_length, batch_size, trg_vocab_size).to(self.device)
        hidden, cell = self.encoder(src)

        input = trg[0]  # shape = [1, batch_size]

        for t in range(1, trg_length):
            output, hidden, cell = self.decoder(input, hidden, cell)
            # output = [batch size, output dim]
            # hidden = [n layers, batch size, hidden dim]
            # cell = [n layers, batch size, hidden dim]
            outputs[t] = output
            teacher_force  = random.random() < teacher_forcing_ratio

            top1 = output.argmax(1)
            input = trg[1] if teacher_force else top1

        return outputs


In [25]:
input_dim = len(de_vocab)
output_dim = len(en_vocab)
encoder_embedding_dim = 256
decoder_embedding_dim = 256
hidden_dim = 512
n_layers = 2
encoder_dropout = 0.5
decoder_dropout = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = Encoder(
    input_dim,
    encoder_embedding_dim,
    hidden_dim,
    n_layers,
    encoder_dropout,
)

decoder = Decoder(
    output_dim,
    decoder_embedding_dim,
    hidden_dim,
    n_layers,
    decoder_dropout,
)

model = Seq2Seq(encoder, decoder, device).to(device)

In [26]:
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)


model.apply(init_weights)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(7853, 256)
    (rnn): LSTM(256, 512, num_layers=2, dropout=0.5)
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(5893, 256)
    (rnn): LSTM(256, 512, num_layers=2, dropout=0.5)
    (fc_out): Linear(in_features=512, out_features=5893, bias=True)
    (dropout): Dropout(p=0.5, inplace=False)
  )
)

In [27]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f"The model has {count_parameters(model):,} trainable parameters")

The model has 13,898,501 trainable parameters


In [28]:
optimizer = optim.Adam(model.parameters())

In [29]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_index)

In [30]:
def train_fn(
    model, data_loader, optimizer, criterion, clip, teacher_forcing_ratio, device
):
    model.train()
    epoch_loss = 0
    for i, batch in enumerate(data_loader):
        src = batch["de_ids"].to(device)
        trg = batch["en_ids"].to(device)
        # src = [src length, batch size]
        # trg = [trg length, batch size]
        optimizer.zero_grad()
        output = model(src, trg, teacher_forcing_ratio)
        # output = [trg length, batch size, trg vocab size]
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        # output = [(trg length - 1) * batch size, trg vocab size]
        trg = trg[1:].view(-1)
        # trg = [(trg length - 1) * batch size]
        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(data_loader)

In [31]:
pip install numpy==1.26.4


In [32]:
def evaluate_fn(model, data_loader, criterion, device):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            src = batch["de_ids"].to(device)
            trg = batch["en_ids"].to(device)
            # src = [src length, batch size]
            # trg = [trg length, batch size]
            output = model(src, trg, 0)  # turn off teacher forcing
            # output = [trg length, batch size, trg vocab size]
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            # output = [(trg length - 1) * batch size, trg vocab size]
            trg = trg[1:].view(-1)
            # trg = [(trg length - 1) * batch size]
            loss = criterion(output, trg)
            epoch_loss += loss.item()
    return epoch_loss / len(data_loader)

In [33]:
n_epochs = 10
clip = 1.0
teacher_forcing_ratio = 0.5

best_valid_loss = float("inf")

for epoch in tqdm.tqdm(range(n_epochs)):
    train_loss = train_fn(
        model,
        train_data_loader,
        optimizer,
        criterion,
        clip,
        teacher_forcing_ratio,
        device,
    )
    valid_loss = evaluate_fn(
        model,
        valid_data_loader,
        criterion,
        device,
    )
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "tut1-model.pt")
    print(f"\tTrain Loss: {train_loss:7.3f} | Train PPL: {np.exp(train_loss):7.3f}")
    print(f"\tValid Loss: {valid_loss:7.3f} | Valid PPL: {np.exp(valid_loss):7.3f}")

 10%|█         | 1/10 [00:48<07:17, 48.56s/it]

	Train Loss:   5.113 | Train PPL: 166.203
	Valid Loss:   4.795 | Valid PPL: 120.865


 20%|██        | 2/10 [01:32<06:08, 46.06s/it]

	Train Loss:   4.697 | Train PPL: 109.569
	Valid Loss:   4.516 | Valid PPL:  91.446


 30%|███       | 3/10 [02:17<05:16, 45.26s/it]

...

	Train Loss:   3.649 | Train PPL:  38.455
	Valid Loss:   3.743 | Valid PPL:  42.245


 90%|█████████ | 9/10 [06:41<00:44, 44.08s/it]

	Train Loss:   3.518 | Train PPL:  33.718
	Valid Loss:   3.672 | Valid PPL:  39.324


100%|██████████| 10/10 [07:25<00:00, 44.52s/it]

	Train Loss:   3.403 | Train PPL:  30.056
	Valid Loss:   3.619 | Valid PPL:  37.306


In [36]:
model.load_state_dict(torch.load("tut1-model.pt"))

test_loss = evaluate_fn(model, test_data_loader, criterion, device)

print(f"| Test Loss: {test_loss:.3f} | Test PPL: {np.exp(test_loss):7.3f} |")

| Test Loss: 3.600 | Test PPL:  36.584 |


In [37]:
def translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
    max_output_length=25,
):
    model.eval()
    with torch.no_grad():
        if isinstance(sentence, str):
            tokens = [token.text for token in de_nlp.tokenizer(sentence)]
        else:
            tokens = [token for token in sentence]
        if lower:
            tokens = [token.lower() for token in tokens]
        tokens = [sos_token] + tokens + [eos_token]
        ids = de_vocab.lookup_indices(tokens)
        tensor = torch.LongTensor(ids).unsqueeze(-1).to(device)
        hidden, cell = model.encoder(tensor)
        inputs = en_vocab.lookup_indices([sos_token])
        for _ in range(max_output_length):
            inputs_tensor = torch.LongTensor([inputs[-1]]).to(device)
            output, hidden, cell = model.decoder(inputs_tensor, hidden, cell)
            predicted_token = output.argmax(-1).item()
            inputs.append(predicted_token)
            if predicted_token == en_vocab[eos_token]:
                break
        tokens = en_vocab.lookup_tokens(inputs)
    return tokens

In [38]:
sentence = test_data[0]["de"]
expected_translation = test_data[0]["en"]

sentence, expected_translation

('Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.',
 'A man in an orange hat starring at something.')

In [39]:
translation = translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
)

In [40]:
translation

['<sos>', 'a', 'man', 'wearing', 'a', 'orange', 'hat', 'is', '.', '.', '<eos>']

In [41]:
sentence = "Ein Mann sitzt auf einer Bank."

In [42]:
translation = translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
)

In [43]:
translation

['<sos>', 'a', 'sitting', 'sitting', 'on', 'a', 'bench', '.', '<eos>']

In [44]:
translations = [
    translate_sentence(
        example["de"],
        model,
        en_nlp,
        de_nlp,
        en_vocab,
        de_vocab,
        lower,
        sos_token,
        eos_token,
        device,
    )
    for example in tqdm.tqdm(test_data)
]

100%|██████████| 1000/1000 [00:08<00:00, 112.41it/s]


In [ ]:
! pip install evaluate

In [47]:
import evaluate

In [ ]:
bleu = evaluate.load("bleu")

In [49]:
predictions = [" ".join(translation[1:-1]) for translation in translations]

references = [[example["en"]] for example in test_data]

In [50]:
predictions[0], references[0]

('a man wearing a orange hat is . .',
 ['A man in an orange hat starring at something.'])

In [51]:
def get_tokenizer_fn(nlp, lower):
    def tokenizer_fn(s):
        tokens = [token.text for token in nlp.tokenizer(s)]
        if lower:
            tokens = [token.lower() for token in tokens]
        return tokens

    return tokenizer_fn

In [52]:
tokenizer_fn = get_tokenizer_fn(en_nlp, lower)

In [53]:
tokenizer_fn(predictions[0]), tokenizer_fn(references[0][0])

(['a', 'man', 'wearing', 'a', 'orange', 'hat', 'is', '.', '.'],
 ['a', 'man', 'in', 'an', 'orange', 'hat', 'starring', 'at', 'something', '.'])

In [54]:
results = bleu.compute(
    predictions=predictions, references=references, tokenizer=tokenizer_fn
)

In [55]:
results

{'bleu': 0.09181094022229538,
 'precisions': [0.42289249806651197,
  0.13419949706621961,
  0.055992680695333946,
  0.02326283987915408],
 'brevity_penalty': 0.9901493797265598,
 'length_ratio': 0.9901975800275693,
 'translation_length': 12930,
 'reference_length': 13058}